# TP 2 : notebook de rattrapage Databricks

- le volume `workspace.shopflow.raw` avec les fichiers Parquet ShopFlow (facteur d'échelle SF1, seed 42) dans `raw/sf1/<table>/part-AAAA-MM.parquet` ;
- les tables `customers`, `products`, `orders`, `order_items`, `payments`, `clickstream` dans `workspace.shopflow`.

Ce notebook recrée exactement cet état. Les données sont **identiques** à celles que vous avez utilisées dans BigQuery : même générateur, même graine.

**Avant de lancer :**

1. Dans *Workspace*, créez un dossier `tp2` (ou utilisez celui du TP 2).
2. Importez-y le fichier `shopflow_gen.py` fourni par l'enseignant (*Import > File*).
3. Importez ce notebook **dans le même dossier**.
4. Attachez le notebook au calcul **Serverless**, puis cliquez sur **Run all**.


## 1. Paramètres


In [ ]:
CATALOG = "workspace"
SCHEMA = "shopflow"
VOLUME = "raw"
SF = 1          # facteur d'échelle du TP 1
SEED = 42       # graine du TP 1 : mêmes données que dans BigQuery

RAW = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/sf1"
TABLES = ["customers", "products", "orders", "order_items", "payments", "clickstream"]

# Nombres de lignes attendus (tableau de la partie 0 du TP 1)
ATTENDU = {
    "customers": 50_001,
    "products": 2_000,
    "orders": 1_011_946,
    "order_items": 1_818_927,
    "payments": 1_031_980,
    "clickstream": 15_984_193,
}

## 2. Schéma et volume

Un **volume** est un espace de fichiers géré par Unity Catalog : c'est là que vont les fichiers Parquet bruts. Les tables, elles, vont dans le **schéma** `shopflow`.

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.shopflow;
CREATE VOLUME IF NOT EXISTS workspace.shopflow.raw;


## 3. Générer les fichiers Parquet dans le volume via le générateur `shopflow_gen.py`


In [ ]:
import sys
sys.path.append("/Workspace/Users/<votre_mail>/tp1")
import shopflow_gen
counts = shopflow_gen.generate(out="/Volumes/workspace/shopflow/raw/sf1", sf=1, seed=42)
counts


## 5. Créer les tables Delta

Une table Delta par source, lue depuis les fichiers Parquet avec `read_files` (même code que la partie 1.B du TP 1). `CREATE OR REPLACE` rend la cellule rejouable sans créer de doublons.

Ces tables ne sont **pas partitionnées** : c'est normal, c'est aussi le cas pour vos camarades qui ont fait le TP 1 sur Databricks.

In [ ]:
%sql
CREATE OR REPLACE TABLE workspace.shopflow.payments AS
SELECT * FROM read_files('/Volumes/workspace/shopflow/raw/sf1/payments/', format => 'parquet');
-- même chose pour payments,customers, products, orders, order_items, clickstream